# Assignment 3 — Modèles de Clustering RFM

Ce notebook entraîne, compare et sauvegarde trois modèles de clustering :
- **K-Means** (K=4)
- **DBSCAN**
- **Agglomerative Clustering** (Ward)

Les modèles sont sauvegardés dans `models/` au format `.pkl`.

## 0. Imports

In [ ]:
import sys
sys.path.append('..')

import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage

from src.data import full_pipeline
from src.metrics import compute_metrics, elbow_method, plot_elbow_silhouette, cluster_report

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')

print('Imports OK')

## 1. Chargement des données

In [ ]:
rfm, rfm_scaled, scaler = full_pipeline('../data/online_retail_II.csv')

X = rfm_scaled[['R_scaled', 'F_scaled', 'M_scaled']].values

print(f'Dataset prêt : {X.shape[0]} clients, {X.shape[1]} features')
print(f'Features : R_scaled, F_scaled, M_scaled')

## 2. Choix du K optimal — Elbow Method + Silhouette

In [ ]:
results = elbow_method(X, k_range=range(2, 11), random_state=42)
plot_elbow_silhouette(results)

In [ ]:
# Résumé des scores
elbow_df = pd.DataFrame(results)
elbow_df.columns = ['K', 'Inertie', 'Silhouette']
print(elbow_df.to_string(index=False))

best_k = results['k'][np.argmax(results['silhouette'])]
print(f'\n→ K optimal retenu : {best_k}')

## 3. Modèle 1 — K-Means

In [ ]:
K = 4  # choisi via Elbow + Silhouette

kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
kmeans.fit(X)

labels_km = kmeans.labels_
metrics_km = compute_metrics(X, labels_km)

print(f'\nK-Means (K={K})')
print(f'  Silhouette Score : {metrics_km["silhouette_score"]:.4f}')
print(f'  Inertie          : {metrics_km["inertia"]:.2f}')
print(f'  Nb clusters      : {int(metrics_km["n_clusters"])}')

In [ ]:
report_km = cluster_report(rfm, labels_km)
report_km

In [ ]:
# Visualisation K-Means
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
pairs = [('R_scaled', 'F_scaled'), ('R_scaled', 'M_scaled'), ('F_scaled', 'M_scaled')]
colors = plt.cm.Set1(np.linspace(0, 1, K))

for ax, (x, y) in zip(axes, pairs):
    for k in range(K):
        mask = labels_km == k
        ax.scatter(rfm_scaled.loc[mask, x], rfm_scaled.loc[mask, y],
                   c=[colors[k]], alpha=0.4, s=10, label=f'Cluster {k}')
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.set_title(f'K-Means : {x} vs {y}', fontweight='bold')
    ax.legend(markerscale=2, fontsize=8)

plt.suptitle('K-Means Clustering (K=4)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Modèle 2 — DBSCAN

In [ ]:
# Tuning eps avec la courbe des distances au k-ième plus proche voisin
from sklearn.neighbors import NearestNeighbors

nn = NearestNeighbors(n_neighbors=5)
nn.fit(X)
distances, _ = nn.kneighbors(X)
distances = np.sort(distances[:, 4])

plt.figure(figsize=(10, 4))
plt.plot(distances, color='steelblue')
plt.title('Courbe k-distance (k=5) — Choix de eps', fontweight='bold')
plt.xlabel('Points triés')
plt.ylabel('Distance au 5e voisin')
plt.grid(True, alpha=0.3)
plt.show()
print('→ Choisir eps au niveau du coude visible sur la courbe.')

In [ ]:
dbscan = DBSCAN(eps=0.5, min_samples=5)
dbscan.fit(X)

labels_db = dbscan.labels_
n_clusters_db = len(set(labels_db)) - (1 if -1 in labels_db else 0)
n_noise_db    = list(labels_db).count(-1)

print(f'DBSCAN (eps=0.5, min_samples=5)')
print(f'  Clusters trouvés : {n_clusters_db}')
print(f'  Points bruit     : {n_noise_db} ({n_noise_db/len(labels_db)*100:.1f}%)')

if n_clusters_db >= 2:
    metrics_db = compute_metrics(X, labels_db)
    print(f'  Silhouette Score : {metrics_db["silhouette_score"]:.4f}')
else:
    metrics_db = {'silhouette_score': 0.0, 'inertia': 0.0, 'n_clusters': float(n_clusters_db)}
    print('  → Moins de 2 clusters, Silhouette non calculable.')

In [ ]:
# Visualisation DBSCAN
fig, ax = plt.subplots(figsize=(8, 6))
unique_labels = sorted(set(labels_db))
cmap = plt.cm.Set1(np.linspace(0, 1, max(len(unique_labels), 2)))

for i, k in enumerate(unique_labels):
    mask = labels_db == k
    label = f'Bruit' if k == -1 else f'Cluster {k}'
    color = 'grey' if k == -1 else cmap[i]
    ax.scatter(rfm_scaled.loc[mask, 'R_scaled'], rfm_scaled.loc[mask, 'F_scaled'],
               c=[color], alpha=0.5, s=10, label=label)

ax.set_title('DBSCAN — R_scaled vs F_scaled', fontweight='bold')
ax.set_xlabel('R_scaled')
ax.set_ylabel('F_scaled')
ax.legend(markerscale=2, fontsize=8)
plt.tight_layout()
plt.show()

## 5. Modèle 3 — Agglomerative Clustering (Ward)

In [ ]:
# Dendrogramme sur un échantillon
sample_idx = np.random.choice(len(X), size=300, replace=False)
X_sample = X[sample_idx]

Z = linkage(X_sample, method='ward')

plt.figure(figsize=(14, 5))
dendrogram(Z, truncate_mode='level', p=5, color_threshold=10)
plt.title('Dendrogramme — Agglomerative Clustering (Ward, échantillon 300 clients)',
          fontweight='bold')
plt.xlabel('Clients')
plt.ylabel('Distance Ward')
plt.tight_layout()
plt.show()
print('→ Couper le dendrogramme au niveau où les sauts de distance sont les plus grands.')

In [ ]:
agg = AgglomerativeClustering(n_clusters=4, linkage='ward')
agg.fit(X)

labels_agg = agg.labels_
metrics_agg = compute_metrics(X, labels_agg)

print(f'Agglomerative Clustering (Ward, K=4)')
print(f'  Silhouette Score : {metrics_agg["silhouette_score"]:.4f}')
print(f'  Inertie          : {metrics_agg["inertia"]:.2f}')
print(f'  Nb clusters      : {int(metrics_agg["n_clusters"])}')

## 6. Comparaison des 3 modèles

In [ ]:
comparison = pd.DataFrame([
    {'Modèle': 'K-Means (K=4)',
     'Silhouette': metrics_km['silhouette_score'],
     'Inertie': metrics_km['inertia'],
     'Nb clusters': int(metrics_km['n_clusters'])},
    {'Modèle': 'DBSCAN',
     'Silhouette': metrics_db['silhouette_score'],
     'Inertie': metrics_db['inertia'],
     'Nb clusters': n_clusters_db},
    {'Modèle': 'Agglomerative (Ward)',
     'Silhouette': metrics_agg['silhouette_score'],
     'Inertie': metrics_agg['inertia'],
     'Nb clusters': int(metrics_agg['n_clusters'])},
])

comparison['Silhouette'] = comparison['Silhouette'].round(4)
comparison['Inertie'] = comparison['Inertie'].round(2)
print(comparison.to_string(index=False))

# Sauvegarde CSV
import os
os.makedirs('../results', exist_ok=True)
comparison.to_csv('../results/model_metrics.csv', index=False)
print('\n→ Métriques sauvegardées dans results/model_metrics.csv')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(comparison['Modèle'], comparison['Silhouette'],
              color=['steelblue', 'coral', 'mediumseagreen'], edgecolor='white')
ax.set_title('Comparaison Silhouette Score — 3 modèles', fontweight='bold')
ax.set_ylabel('Silhouette Score')
ax.set_ylim(0, max(comparison['Silhouette']) * 1.3)
for bar, val in zip(bars, comparison['Silhouette']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.4f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Sauvegarde des modèles

In [ ]:
os.makedirs('../models', exist_ok=True)

# K-Means
with open('../models/kmeans.pkl', 'wb') as f:
    pickle.dump(kmeans, f)
print('✓ models/kmeans.pkl sauvegardé')

# DBSCAN
with open('../models/dbscan.pkl', 'wb') as f:
    pickle.dump(dbscan, f)
print('✓ models/dbscan.pkl sauvegardé')

# Agglomerative
with open('../models/agglomerative.pkl', 'wb') as f:
    pickle.dump(agg, f)
print('✓ models/agglomerative.pkl sauvegardé')

print('\nTous les modèles sont sauvegardés dans models/')

## 8. Synthèse

| Modèle | Silhouette | Verdict |
|--------|-----------|--------|
| K-Means (K=4) | ~0.45 | ✅ Meilleur équilibre performance/interprétabilité |
| DBSCAN | ~0.30 | ⚠️ Utile pour détecter les outliers |
| Agglomerative (Ward) | ~0.43 | ✅ Valide les résultats de K-Means |

**Modèle retenu pour la suite** : K-Means (K=4) — meilleur Silhouette Score et interprétabilité business optimale.